In [1]:
import matplotlib.pyplot as plt
import os
import glob
import pandas_datareader.data as web
import requests
import datetime
import pandas as pd
from dbnomics import fetch_series
from utils import fetch_imf_data, fetch_imf_bulk, fetch_imf_api, fetch_imts, _parse_imf_sdmx_json
from config import country_dict, us_economy_metrics, country_dict_3_char, dict_3_char, dict_2_char


In [2]:
# Paths
imf_cpi_path = "data/imf_cpi.csv"
imf_gdp_path = "data/imf_gdp.csv"
imf_unemployment_path = "data/imf_unemployment_rate.csv"
imf_industrial_production_index_path = "data/imf_industrial_production_index.csv"
imf_exports_path = "data/imf_exports.csv"
imf_imports_path = "data/imf_imports.csv"
us_economy_path = "data/us_economy.csv"
cb_policy_rates_path = "data/cb_policy_rates.csv"
global_manufacturing_PMI_path = "data/global_manufacturing_PMI.csv"
im_ex_path = "data/im_ex/"
im_path = "data/im/"
ex_path = "data/ex/"
retail_sales_path = "data/retail_sales.csv"
country_mapping_path = "data/d_country.csv"
exchange_rate_path = "data/exchange_rate.csv"

target_countries = ["VN", "TH", "SG", "ID", "MY", "PH", "BN", "KH", "LA", "MM", "TL", "DE", "FR", "IT", "ES", "NL", "GB", "BE", "AT", "PT", "GR", "FI", "IE", "DK", "SE", "KW", "OM", "BH"]


In [12]:
# Tạo DataFrame Mapping
df_map_2_char = pd.DataFrame(list(dict_2_char.items()), columns=['iso2_code', 'country_name_2c'])
df_map_3_char = pd.DataFrame(list(dict_3_char.items()), columns=['iso3_code', 'country_name'])

# Ghép theo index để tạo bảng mapping 3 cột chuẩn
df_country_mapping = pd.concat([df_map_2_char[['iso2_code']], df_map_3_char], axis=1)

# Reorder cột cho đẹp: iso2_code | iso3_code | country_name
df_country_mapping = df_country_mapping[['iso2_code', 'iso3_code', 'country_name']]

df_country_mapping.to_csv(country_mapping_path)

In [13]:
df_country_mapping

,iso2_code,iso3_code,country_name
0,VN,VNM,Vietnam
1,TH,THA,Thailand
2,SG,SGP,Singapore
3,ID,IDN,Indonesia
4,MY,MYS,Malaysia
...,...,...,...
72,VU,VUT,Vanuatu
73,WS,WSM,Samoa
74,ST,STP,Sao Tome and Principe
75,SUH,SUN,Former U.S.S.R.


In [3]:
exchange_rate = fetch_series(
    provider_code="IMF", 
    dataset_code="IFS", # Exchange Rates, US Dollar per Domestic Currency, Period Average, Rate
    max_nb_series = 500000,
    dimensions={
        "FREQ": ["M"],
        # "REF_AREA": list(country_dict.keys())
        "INDICATOR": ['EDNA_USD_XDC_RATE']
    }
)
exchange_rate.to_csv(exchange_rate_path)
exchange_rate_summary = exchange_rate.copy()
exchange_rate_summary['year'] = pd.to_datetime(exchange_rate_summary['period'], errors='coerce').dt.strftime("%Y")
exchange_rate_summary = exchange_rate_summary.groupby(['year', 'REF_AREA'], as_index=False)['value'].mean()

In [30]:
# Pillar 1: System Health & Growth
# 1. REAL GDP GROWTH (Tăng trưởng GDP Thực tế - Hàng Quý 'Q' hoặc Hàng Năm 'A')
# print("Fetching Real GDP Growth Data...")
# imf_gdp = fetch_imf_data(
#     country_dict=country_dict, 
#     dataset_code = "IFS",
#     # metric_suffix="NGDP_R_XDC", #  Real
#     metric_suffix="NGDP_​XDC", # Nominal
#     frequency="A"
# )
imf_gdp = fetch_series(
    provider_code="IMF", 
    dataset_code="IFS", 
    max_nb_series=50000000000,
    dimensions={
        "FREQ": ["A"],
        "INDICATOR": ['NGDP_XDC']
    }
).dropna(subset=['value'])

# Add exchange rate
imf_gdp = imf_gdp.merge(exchange_rate_summary, left_on=['REF_AREA', 'original_period'], right_on=['REF_AREA', 'year'], how='left')
imf_gdp = imf_gdp.rename(columns={
        "value_x": "value",
        'value_y': 'usd/local'
})
imf_gdp = imf_gdp[['@frequency', 'provider_code', 'dataset_code', 'dataset_name',
       'series_code', 'series_name', 'original_period', 'period',
       'original_value', 'value', 'FREQ', 'REF_AREA', 'INDICATOR', 'Frequency',
       'Reference Area', 'Indicator', "usd/local"]]

imf_gdp['local/usd'] = 1/imf_gdp['usd/local']
imf_gdp['value_usd'] = imf_gdp['value']*imf_gdp['usd/local']
imf_gdp.to_csv(imf_gdp_path, index=False)

In [32]:
imf_gdp[imf_gdp['REF_AREA']=='VN']

,@frequency,provider_code,dataset_code,dataset_name,series_code,series_name,original_period,period,original_value,value,FREQ,REF_AREA,INDICATOR,Frequency,Reference Area,Indicator,usd/local,local/usd,value_usd
5882,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2000,2000-01-01,4.416461e+08,4.416461e+08,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000071,14166.050223,31176.377285
5883,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2001,2001-01-01,4.812946e+08,4.812946e+08,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000068,14719.753952,32697.191065
5884,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2002,2002-01-01,5.357618e+08,5.357618e+08,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000065,15278.999804,35065.243204
5885,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2003,2003-01-01,6.134430e+08,6.134430e+08,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000064,15509.233152,39553.407938
5886,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2004,2004-01-01,7.793377e+08,7.793377e+08,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000064,15745.933715,49494.535197
5887,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2005,2005-01-01,9.140012e+08,9.140012e+08,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000063,15858.821839,57633.610880
5888,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2006,2006-01-01,1.061565e+09,1.061565e+09,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000063,15994.020474,66372.587212
5889,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2007,2007-01-01,1.246769e+09,1.246769e+09,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000062,16104.774974,77416.126027
5890,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2008,2008-01-01,1.616047e+09,1.616047e+09,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000061,16298.799524,99151.298019
5891,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_XDC,"Annual – Viet Nam – Gross Domestic Product, No...",2009,2009-01-01,1.809149e+09,1.809149e+09,A,VN,NGDP_XDC,Annual,Viet Nam,"Gross Domestic Product, Nominal, Domestic Curr...",0.000059,17060.953286,106040.320237


In [ ]:
imf_gdp = imf_gdp.merge(exchange_rate_summary, left_on=['REF_AREA', 'original_period'], right_on=['REF_AREA', 'year'], how='left')
imf_gdp = imf_gdp.rename({
        "value_x": "value",
        'value_y': 'exchange_rate'
})
imf_gdp = imf_gdp[['@frequency', 'provider_code', 'dataset_code', 'dataset_name',
       'series_code', 'series_name', 'original_period', 'period',
       'original_value', 'value', 'FREQ', 'REF_AREA', 'INDICATOR', 'Frequency',
       'Reference Area', 'Indicator', 'country_name', 'country_code',
       'value_yoy', "exchange_rate"]]

,@frequency,provider_code,dataset_code,dataset_name,series_code,series_name,original_period,period,original_value,value_x,...,REF_AREA,INDICATOR,Frequency,Reference Area,Indicator,country_name,country_code,value_yoy,year,value_y
0,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_R_XDC,"Annual – Viet Nam – Gross Domestic Product, Re...",2004,2004-01-01,1.477161e+09,1.477161e+09,...,VN,NGDP_R_XDC,Annual,Viet Nam,"Gross Domestic Product, Real, Domestic Currency",Viet Nam,VN,NaN,2004,0.000064
1,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_R_XDC,"Annual – Viet Nam – Gross Domestic Product, Re...",2005,2005-01-01,1.588646e+09,1.588646e+09,...,VN,NGDP_R_XDC,Annual,Viet Nam,"Gross Domestic Product, Real, Domestic Currency",Viet Nam,VN,NaN,2005,0.000063
2,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_R_XDC,"Annual – Viet Nam – Gross Domestic Product, Re...",2006,2006-01-01,1.699501e+09,1.699501e+09,...,VN,NGDP_R_XDC,Annual,Viet Nam,"Gross Domestic Product, Real, Domestic Currency",Viet Nam,VN,NaN,2006,0.000063
3,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_R_XDC,"Annual – Viet Nam – Gross Domestic Product, Re...",2007,2007-01-01,1.820667e+09,1.820667e+09,...,VN,NGDP_R_XDC,Annual,Viet Nam,"Gross Domestic Product, Real, Domestic Currency",Viet Nam,VN,NaN,2007,0.000062
4,annual,IMF,IFS,International Financial Statistics (IFS),A.VN.NGDP_R_XDC,"Annual – Viet Nam – Gross Domestic Product, Re...",2008,2008-01-01,1.923749e+09,1.923749e+09,...,VN,NGDP_R_XDC,Annual,Viet Nam,"Gross Domestic Product, Real, Domestic Currency",Viet Nam,VN,0.302329,2008,0.000061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1967,annual,IMF,IFS,International Financial Statistics (IFS),A.U2.NGDP_R_XDC,Annual – Euro area (Member States and Institut...,2020,2020-01-01,1.015642e+07,1.015642e+07,...,U2,NGDP_R_XDC,Annual,Euro area (Member States and Institutions of t...,"Gross Domestic Product, Real, Domestic Currency",Euro Area (Member States and Institutions of t...,U2,-0.002718,2020,1.141282
1968,annual,IMF,IFS,International Financial Statistics (IFS),A.U2.NGDP_R_XDC,Annual – Euro area (Member States and Institut...,2021,2021-01-01,1.079630e+07,1.079630e+07,...,U2,NGDP_R_XDC,Annual,Euro area (Member States and Institutions of t...,"Gross Domestic Product, Real, Domestic Currency",Euro Area (Member States and Institutions of t...,U2,0.033308,2021,1.183527
1969,annual,IMF,IFS,International Financial Statistics (IFS),A.U2.NGDP_R_XDC,Annual – Euro area (Member States and Institut...,2022,2022-01-01,1.117319e+07,1.117319e+07,...,U2,NGDP_R_XDC,Annual,Euro area (Member States and Institutions of t...,"Gross Domestic Product, Real, Domestic Currency",Euro Area (Member States and Institutions of t...,U2,0.050798,2022,1.053877
1970,annual,IMF,IFS,International Financial Statistics (IFS),A.U2.NGDP_R_XDC,Annual – Euro area (Member States and Institut...,2023,2023-01-01,1.123542e+07,1.123542e+07,...,U2,NGDP_R_XDC,Annual,Euro area (Member States and Institutions of t...,"Gross Domestic Product, Real, Domestic Currency",Euro Area (Member States and Institutions of t...,U2,0.039830,2023,1.081580


In [4]:
# Pillar 1: System Health & Growth
# 2. INDUSTRIAL PRODUCTION INDEX - IPI (Thay đổi dataset_code="IFS")
ipi_raw = fetch_imf_bulk(
    dataset_code="IFS",
    metric_code="AIP_IX", 
    frequency="M", 
    country_list=target_countries
)
if not ipi_raw.empty:
    ipi_raw.dropna(subset=['value']).to_csv(imf_industrial_production_index_path, index=False)

--- Đang tải dữ liệu cho mã: AIP_IX (M) ---
✅ Tải thành công 11153 dòng dữ liệu!


In [5]:
# Pillar 2: Price Stability & Monetary Policy
# 3. CPI
imf_cpi = fetch_imf_data(country_dict=country_dict, dataset_code = "CPI", metric_suffix="PCPI_IX", frequency="M").dropna(subset=['value'])
# Show 10 samples
imf_cpi.head(10)
# Export data
imf_cpi.to_csv(imf_cpi_path)

Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.VA.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.TM.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.TV.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.VU.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.SUH.PCPI_IX'}
Could not load series: {'dataset_code': 'CPI', 'message': 'Could not load series', 'provider_code': 'IMF', 'series_code': 'M.U2.PCPI_IX'}


In [6]:
# Pillar 2: Price Stability & Monetary Policy
# 4. Central bank policy rates
cb_policy_rates = fetch_series(
    provider_code="BIS", 
    dataset_code="WS_CBPOL", 
    dimensions={
        "FREQ": ["D"],
        "REF_AREA": list(country_dict.keys())
    }
)
cb_policy_rates.to_csv(cb_policy_rates_path)

In [7]:
# Pillar 3: Labor Market & Consumer Market
# 5. Unemployment Rate
unemployment_raw = fetch_series(
    provider_code="IMF", 
    dataset_code="WEO:latest", 
    dimensions={
        "weo-country": list(country_dict_3_char.keys()),
        "weo-subject": ["LUR"],
        "unit": ["pcent_total_labor_force"]
    }
)
unemployment_raw.to_csv(imf_unemployment_path)

In [8]:
# Pillar 3: Labor Market & Consumer Market
# 6. Retail Sales Growth
# DBnomics: OECD (KEI).
retail_sales_raw_set_1 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],     #  Sales > Retail trade > Total retail trade > Value
        "MEASURE": ["IXEB"],    # US Dollars, monthly level
        "FREQUENCY": ["M"],     # Lấy theo Tháng (Monthly) để monitor cực nhạy
        "LOCATION": ['AUT', 'BEL', 'BGR', 'CHE', 'ESP', 'EST', 'FIN', 'FRA', 'GRC', 'HRV', 'HUN', 'IRL', 'ITA', 'LTU', 'NLD', 'NOR', 'PRT', 'SVK', 'TUR']
    }
)
retail_sales_raw_set_2 = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["SLRTTO02"],
        "MEASURE": ["ML"],
        "FREQUENCY": ["M"],
        "LOCATION": ['CHN', 'RUS', 'USA', 'ZAF']
    }
)
retail_sales_raw = pd.concat([retail_sales_raw_set_1, retail_sales_raw_set_2], ignore_index=True)
retail_sales_raw.to_csv(retail_sales_path, encoding="utf-8-sig")

In [9]:
# Pillar 4: Trade & Geopolitics
# 7. Global Manufacturing PMI
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.
global_manufacturing_PMI = fetch_series(
    provider_code="OECD", 
    dataset_code="MEI", 
    dimensions={
        "SUBJECT": ["BSCICP02"], 
        "FREQUENCY": ["M"],
    }
)
global_manufacturing_PMI.to_csv(global_manufacturing_PMI_path)

In [ ]:
# # Pillar 4: Trade & Geopolitics
# # 8.1. Merchandise Trade Balance

# # ==========================================
# # 1. CẤU HÌNH & THIẾT LẬP
# # ==========================================
# # Đặt True khi muốn CẬP NHẬT DỮ LIỆU MỚI
# # Đặt False khi bị CRASH/NGẮT MẠNG và muốn CHẠY NỐI TIẾP
# FORCE_UPDATE = True  

# im_ex_path = "data/im_ex"
# os.makedirs(im_ex_path, exist_ok=True)

# country_codes = list(country_dict.keys())
# CHUNK_SIZE = 10

# # ==========================================
# # 2. TIẾN HÀNH TẢI DỮ LIỆU THEO CHUNK
# # ==========================================
# print(f"🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = {FORCE_UPDATE})...\n")

# for i in range(0, len(country_codes), CHUNK_SIZE):
#     chunk_codes = country_codes[i : i + CHUNK_SIZE]
#     batch_num = (i // CHUNK_SIZE) + 1
#     file_path = os.path.join(im_ex_path, f"im_ex_batch_{batch_num}.csv")
    
#     # Kiểm tra điều kiện bỏ qua nếu file đã tồn tại và không ép buộc update
#     if not FORCE_UPDATE and os.path.exists(file_path):
#         print(f"⏩ Nhóm {batch_num} ({len(chunk_codes)} nước) đã có file -> Bỏ qua.")
#         continue

#     print(f"🔄 Đang tải nhóm {batch_num}/{-(len(country_codes) // -CHUNK_SIZE)} ({len(chunk_codes)} nước): {chunk_codes}...")
    
#     try:
#         df_chunk = fetch_series(
#             provider_code="IMF", 
#             dataset_code="DOT",
#             max_nb_series=53188,
#             dimensions={
#                 "FREQ": ["M"],
#                 "REF_AREA": chunk_codes,
#                 "INDICATOR": ["TBG_USD"]
#             }
#         )
        
#         if df_chunk is not None and not df_chunk.empty:
#             df_chunk = df_chunk[df_chunk['value'].notna()]
#             df_chunk.to_csv(file_path, index=False, encoding="utf-8-sig")
#             print(f"   => ✅ Đã lưu: {file_path}")
#         else:
#             print(f"   => ⚠️ Nhóm {batch_num} không phản hồi dữ liệu.")
            
#     except Exception as e:
#         print(f"   => ❌ Lỗi ở nhóm {batch_num}: {e}")

# print("\n🎉 Hoàn thành xong bước tải dữ liệu!")

🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = True)...

🔄 Đang tải nhóm 1/8 (10 nước): ['VN', 'TH', 'SG', 'ID', 'MY', 'PH', 'BN', 'KH', 'LA', 'MM']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_1.csv
🔄 Đang tải nhóm 2/8 (10 nước): ['TL', 'DE', 'FR', 'IT', 'ES', 'NL', 'GB', 'BE', 'AT', 'PT']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_2.csv
🔄 Đang tải nhóm 3/8 (10 nước): ['GR', 'FI', 'IE', 'DK', 'SE', 'PL', 'CZ', 'RO', 'HU', 'VA']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_3.csv
🔄 Đang tải nhóm 4/8 (10 nước): ['UA', 'TR', 'XK', 'US', 'CA', 'MX', 'BR', 'AR', 'CO', 'SR']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_4.csv
🔄 Đang tải nhóm 5/8 (10 nước): ['SV', 'SX', 'TT', 'UY', 'VC', 'VE', 'SA', 'AE', 'QA', 'KW']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_5.csv
🔄 Đang tải nhóm 6/8 (10 nước): ['OM', 'BH', 'IQ', 'SY', 'TJ', 'TM', 'UZ', 'YE', 'SN', 'SO']...
   => ✅ Đã lưu: data/im_ex\im_ex_batch_6.csv
🔄 Đang tải nhóm 7/8 (10 nước): ['SS', 'SZ', 'TD', 'TG', 'TN', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']...
   => ✅ 

In [ ]:
# # Pillar 4: Trade & Geopolitics
# # 8.2. Merchandise Import Value

# # ==========================================
# # 1. CẤU HÌNH & THIẾT LẬP
# # ==========================================
# # Đặt True khi muốn CẬP NHẬT DỮ LIỆU MỚI
# # Đặt False khi bị CRASH/NGẮT MẠNG và muốn CHẠY NỐI TIẾP
# FORCE_UPDATE = True  

# os.makedirs(im_path, exist_ok=True)

# country_codes = list(country_dict.keys())
# CHUNK_SIZE = 10

# # ==========================================
# # 2. TIẾN HÀNH TẢI DỮ LIỆU THEO CHUNK
# # ==========================================
# print(f"🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = {FORCE_UPDATE})...\n")

# for i in range(0, len(country_codes), CHUNK_SIZE):
#     chunk_codes = country_codes[i : i + CHUNK_SIZE]
#     batch_num = (i // CHUNK_SIZE) + 1
#     file_path = os.path.join(im_path, f"im_batch_{batch_num}.csv")
    
#     # Kiểm tra điều kiện bỏ qua nếu file đã tồn tại và không ép buộc update
#     if not FORCE_UPDATE and os.path.exists(file_path):
#         print(f"⏩ Nhóm {batch_num} ({len(chunk_codes)} nước) đã có file -> Bỏ qua.")
#         continue

#     print(f"🔄 Đang tải nhóm {batch_num}/{-(len(country_codes) // -CHUNK_SIZE)} ({len(chunk_codes)} nước): {chunk_codes}...")
    
#     try:
#         df_chunk = fetch_series(
#             provider_code="IMF", 
#             dataset_code="DOT",
#             max_nb_series=53188,
#             dimensions={
#                 "FREQ": ["M"],
#                 "REF_AREA": chunk_codes,
#                 "INDICATOR": ["TMG_FOB_USD"]
#             }
#         )
        
#         if df_chunk is not None and not df_chunk.empty:
#             df_chunk = df_chunk[df_chunk['value'].notna()]
#             df_chunk.to_csv(file_path, index=False, encoding="utf-8-sig")
#             print(f"   => ✅ Đã lưu: {file_path}")
#         else:
#             print(f"   => ⚠️ Nhóm {batch_num} không phản hồi dữ liệu.")
            
#     except Exception as e:
#         print(f"   => ❌ Lỗi ở nhóm {batch_num}: {e}")

# print("\n🎉 Hoàn thành xong bước tải dữ liệu!")

🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = True)...

🔄 Đang tải nhóm 1/8 (10 nước): ['VN', 'TH', 'SG', 'ID', 'MY', 'PH', 'BN', 'KH', 'LA', 'MM']...
   => ⚠️ Nhóm 1 không phản hồi dữ liệu.
🔄 Đang tải nhóm 2/8 (10 nước): ['TL', 'DE', 'FR', 'IT', 'ES', 'NL', 'GB', 'BE', 'AT', 'PT']...
   => ⚠️ Nhóm 2 không phản hồi dữ liệu.
🔄 Đang tải nhóm 3/8 (10 nước): ['GR', 'FI', 'IE', 'DK', 'SE', 'PL', 'CZ', 'RO', 'HU', 'VA']...
   => ✅ Đã lưu: data/im/im_batch_3.csv
🔄 Đang tải nhóm 4/8 (10 nước): ['UA', 'TR', 'XK', 'US', 'CA', 'MX', 'BR', 'AR', 'CO', 'SR']...
   => ✅ Đã lưu: data/im/im_batch_4.csv
🔄 Đang tải nhóm 5/8 (10 nước): ['SV', 'SX', 'TT', 'UY', 'VC', 'VE', 'SA', 'AE', 'QA', 'KW']...
   => ⚠️ Nhóm 5 không phản hồi dữ liệu.
🔄 Đang tải nhóm 6/8 (10 nước): ['OM', 'BH', 'IQ', 'SY', 'TJ', 'TM', 'UZ', 'YE', 'SN', 'SO']...
   => ⚠️ Nhóm 6 không phản hồi dữ liệu.
🔄 Đang tải nhóm 7/8 (10 nước): ['SS', 'SZ', 'TD', 'TG', 'TN', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']...
   => ✅ Đã lưu: data/im/im_batch_7.csv
🔄

In [ ]:
# imf_imports = fetch_imf_api(
#     dataflow_id="IMTS",
#     key="VNM+USA+DEU.MG_FOB_USD.G001.M",  # '+' joins multiple countries in one call
#     start_period="2015-01",
# )

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()
# api_key = os.environ["IMF_API_KEY"]
# IMF_API_BASE = "https://api.imf.org/external/sdmx/3.0/data/dataflow"
# # import_endpoint = 'https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/IMTS/%2B/*.*.*.M?c%5BTIME_PERIOD%5D=ge:1959-12-31+le:2026-03-31&attributes=all&detail=full&includeHistory=true&limit=100'
# params = {
#         "dimensionAtObservation": "TIME_PERIOD",
#         "attributes": "all",
#         "detail":"full",
#         "measures": "all",
#         "includeHistory": "true",
#     }
# agency='IMF.STA'
# dataflow_id='IMTS'
# version='1.0.0'
# key='VNM.XG_FOB_USD.*.M'
# url = f"{IMF_API_BASE}/{agency}/{dataflow_id}/{version}/{key}"
# import_data = requests.get(
#         url,
#         params=params,
#         headers={"Accept": "application/json", "Ocp-Apim-Subscription-Key": api_key},
#     )



In [ ]:
# data = import_data.json()['data']['dataSets'][0]['series']['0:0:0:0']['observations']
# columns = import_data.json()['data']['dataSets'][0]['dimensionGroupAttributes'][':0:::']
# # import_df = pd.DataFrame(data, columns)

In [ ]:
# structure = import_data.json()['data']['structures'][0]
# series_dims = structure["dimensions"]["series"]
# time_values = [v["value"] for v in structure["dimensions"]["observation"][0]["values"]]
# rows = []
# for series_key, series in import_data.json()["data"]["dataSets"][0].get("series", {}).items():
#     dim_indexes = [int(i) for i in series_key.split(":")]
#     dim_values = {
#                 dim["id"]: dim["values"][idx]["id"]
#                 for dim, idx in zip(series_dims, dim_indexes)
#             }

#     for obs_index, obs in series["observations"].items():
#                 rows.append({**dim_values, "period": time_values[int(obs_index)], "value": obs[0]})
# df = pd.DataFrame(rows)
# df

,COUNTRY,INDICATOR,COUNTERPART_COUNTRY,FREQUENCY,period,value
0,VNM,XG_FOB_USD,ABW,M,1996-M01,8833.33333333333
1,VNM,XG_FOB_USD,ABW,M,1996-M02,8833.33333333333
2,VNM,XG_FOB_USD,ABW,M,1996-M03,8833.33333333333
3,VNM,XG_FOB_USD,ABW,M,1996-M04,8833.33333333333
4,VNM,XG_FOB_USD,ABW,M,1996-M05,8833.33333333333
...,...,...,...,...,...,...
53757,VNM,XG_FOB_USD,ZWE,M,2017-M08,1046821
53758,VNM,XG_FOB_USD,ZWE,M,2017-M09,1046821
53759,VNM,XG_FOB_USD,ZWE,M,2017-M10,1046821
53760,VNM,XG_FOB_USD,ZWE,M,2017-M11,1046821


In [ ]:
# Pillar 4: Trade & Geopolitics
# 8.2. Merchandise Import Value
imports_data_1 = fetch_imts("MG_FOB_USD", start_period="2010-01", end_period="2026-03")
imports_data_1['period'] = pd.to_datetime(imports_data_1['period'], format="%Y-M%m").dt.strftime('%Y-%m')
import_data_countries = list(imports_data_1['COUNTRY'].unique())
print(f'fetching the first imports_data for countries: {import_data_countries}')

imports_data_2 = fetch_imts("MG_CIF_USD", start_period="2010-01", end_period="2026-03")
imports_data_2 = imports_data_2[~imports_data_2['COUNTRY'].isin(import_data_countries)]
imports_data_2['period'] = pd.to_datetime(imports_data_2['period'], format="%Y-M%m").dt.strftime('%Y-%m')
print(f"fetching the second imports_data for countries: {imports_data_2['COUNTRY'].unique()}")

imports_data = pd.concat([imports_data_1, imports_data_2], ignore_index=True)
imports_data.to_csv(imf_imports_path, index=False, encoding="utf-8-sig")


fetching the first imports_data for countries: ['AUS', 'BMU', 'BRA', 'CAN', 'DOM', 'MEX', 'PER', 'PNG', 'PRY', 'SLB', 'ZAF', 'ZWE']


In [ ]:
imports_data_2 = imports_data_2[~imports_data_2['COUNTRY'].isin(list(imports_data['COUNTRY'].unique()))]

In [31]:
imports_data_2['COUNTRY'].unique()

<StringArray>
['ABW', 'AFG', 'AGO', 'AIA', 'ALB', 'ANT', 'ARE', 'ARG', 'ARM', 'ASM',
 ...
 'UZB', 'VAT', 'VCT', 'VEN', 'VNM', 'VUT', 'WBG', 'WSM', 'YEM', 'ZMB']
Length: 216, dtype: str

In [ ]:
# Pillar 4: Trade & Geopolitics
# 8.3. Merchandise Export Value
export_data = fetch_imts("XG_FOB_USD", start_period="2010-01", end_period="2026-03")
export_data['period'] = pd.to_datetime(export_data['period'], format="%Y-M%m").dt.strftime('%Y-%m')
export_data.head(10)
export_data.to_csv(imf_exports_path, index=False, encoding="utf-8-sig")


In [ ]:
# # Pillar 4: Trade & Geopolitics
# # 8.3. Merchandise Export Value

# # ==========================================
# # 1. CẤU HÌNH & THIẾT LẬP
# # ==========================================
# # Đặt True khi muốn CẬP NHẬT DỮ LIỆU MỚI
# # Đặt False khi bị CRASH/NGẮT MẠNG và muốn CHẠY NỐI TIẾP
# FORCE_UPDATE = True  

# os.makedirs(ex_path, exist_ok=True)

# country_codes = list(country_dict.keys())
# CHUNK_SIZE = 10

# # ==========================================
# # 2. TIẾN HÀNH TẢI DỮ LIỆU THEO CHUNK
# # ==========================================
# print(f"🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = {FORCE_UPDATE})...\n")

# for i in range(0, len(country_codes), CHUNK_SIZE):
#     chunk_codes = country_codes[i : i + CHUNK_SIZE]
#     batch_num = (i // CHUNK_SIZE) + 1
#     file_path = os.path.join(ex_path, f"ex_batch_{batch_num}.csv")
    
#     # Kiểm tra điều kiện bỏ qua nếu file đã tồn tại và không ép buộc update
#     if not FORCE_UPDATE and os.path.exists(file_path):
#         print(f"⏩ Nhóm {batch_num} ({len(chunk_codes)} nước) đã có file -> Bỏ qua.")
#         continue

#     print(f"🔄 Đang tải nhóm {batch_num}/{-(len(country_codes) // -CHUNK_SIZE)} ({len(chunk_codes)} nước): {chunk_codes}...")
    
#     try:
#         df_chunk = fetch_series(
#             provider_code="IMF", 
#             dataset_code="DOT",
#             max_nb_series=53188,
#             dimensions={
#                 "FREQ": ["M"],
#                 "REF_AREA": chunk_codes,
#                 "INDICATOR": ["TXG_FOB_USD"]
#             }
#         )
        
#         if df_chunk is not None and not df_chunk.empty:
#             df_chunk = df_chunk[df_chunk['value'].notna()]
#             df_chunk.to_csv(file_path, index=False, encoding="utf-8-sig")
#             print(f"   => ✅ Đã lưu: {file_path}")
#         else:
#             print(f"   => ⚠️ Nhóm {batch_num} không phản hồi dữ liệu.")
            
#     except Exception as e:
#         print(f"   => ❌ Lỗi ở nhóm {batch_num}: {e}")

# print("\n🎉 Hoàn thành xong bước tải dữ liệu!")

🚀 Bắt đầu quá trình tải (Chế độ FORCE_UPDATE = True)...

🔄 Đang tải nhóm 1/8 (10 nước): ['VN', 'TH', 'SG', 'ID', 'MY', 'PH', 'BN', 'KH', 'LA', 'MM']...
   => ✅ Đã lưu: data/ex/ex_batch_1.csv
🔄 Đang tải nhóm 2/8 (10 nước): ['TL', 'DE', 'FR', 'IT', 'ES', 'NL', 'GB', 'BE', 'AT', 'PT']...
   => ✅ Đã lưu: data/ex/ex_batch_2.csv
🔄 Đang tải nhóm 3/8 (10 nước): ['GR', 'FI', 'IE', 'DK', 'SE', 'PL', 'CZ', 'RO', 'HU', 'VA']...
   => ✅ Đã lưu: data/ex/ex_batch_3.csv
🔄 Đang tải nhóm 4/8 (10 nước): ['UA', 'TR', 'XK', 'US', 'CA', 'MX', 'BR', 'AR', 'CO', 'SR']...
   => ✅ Đã lưu: data/ex/ex_batch_4.csv
🔄 Đang tải nhóm 5/8 (10 nước): ['SV', 'SX', 'TT', 'UY', 'VC', 'VE', 'SA', 'AE', 'QA', 'KW']...
   => ✅ Đã lưu: data/ex/ex_batch_5.csv
🔄 Đang tải nhóm 6/8 (10 nước): ['OM', 'BH', 'IQ', 'SY', 'TJ', 'TM', 'UZ', 'YE', 'SN', 'SO']...
   => ✅ Đã lưu: data/ex/ex_batch_6.csv
🔄 Đang tải nhóm 7/8 (10 nước): ['SS', 'SZ', 'TD', 'TG', 'TN', 'TZ', 'UG', 'ZA', 'ZM', 'ZW']...
   => ✅ Đã lưu: data/ex/ex_batch_7.csv
🔄 Đan

In [13]:
# Pillar 4: Trade & Geopolitics
# 9. Commodity Price Index
# DBnomics: WB (World Bank - Pink Sheet Commodity Prices) hoặc IMF.

In [14]:
# Timeframe covering Trump baseline through the entirety of the Biden administration
start = datetime.datetime(2000, 1, 1)
end = datetime.datetime(2025, 12, 31)

# Define the exact FRED series codes mapped to your portfolio naming convention
try:
    df_list = []
    for name, code in us_economy_metrics.items():
        print(f"-> Fetching {name} ({code})...")
        # Pull data directly from St. Louis Fed servers
        df = web.DataReader(code, 'fred', start, end)
        # 1. Clean the individual metric's index and rename columns
        df = df.reset_index().rename(columns={'DATE': 'Date', code: 'Value'})
        # 2. Handle frequency alignment: Forward-fill quarterly data before stacking
        # This ensures April and May inherit Q1 data before it gets mixed with monthly metrics
        if name in ['Real_GDP', 'Manufacturing_Investment']:
            # Create a continuous monthly date range to map the quarterly data onto
            monthly_range = pd.date_range(start=start, end=end, freq='MS')
            df = df.set_index('Date').reindex(monthly_range).ffill().reset_index().rename(columns={'index': 'Date'})
        # 3. Add the metadata column so we know which metric this row belongs to
        df['Metric_Name'] = name
        # Reorder columns to look clean: Date | Metric_Name | Value
        df = df[['Date', 'Metric_Name', 'Value']]        
        df_list.append(df)
    print("Consolidating and formatting data frequencies...")
    # Concatenate all tables along the Date axis
    bi_dataset = pd.concat(df_list, axis=0, ignore_index=True)
    # Forward-fill (ffill) the quarterly data (GDP & Investment) so monthly rows aren't blank
    # This prevents relationship errors inside Power BI's model
    bi_dataset = bi_dataset.ffill()
    # Reset index to turn the Date from an index into a normal clean column
    bi_dataset = bi_dataset.reset_index().rename(columns={'DATE': 'Date'})
    # Export to local project directory
    bi_dataset.to_csv(us_economy_path, index=False)
    print(f"Success! Master file saved as '{us_economy_path}'")
    print(bi_dataset.tail(10))

except Exception as e:
    print(f"Pipeline Execution Failed: {e}")

-> Fetching CPI_All_Items (CPIAUCSL)...
-> Fetching Unemployment_Rate (UNRATE)...
-> Fetching Total_Nonfarm_Payrolls (PAYEMS)...
-> Fetching Real_GDP (GDPC1)...
-> Fetching Manufacturing_Investment (C307RX1Q020SBEA)...
Consolidating and formatting data frequencies...
Success! Master file saved as 'data/us_economy.csv'
      index       Date               Metric_Name    Value
1550   1550 2025-03-01  Manufacturing_Investment  145.228
1551   1551 2025-04-01  Manufacturing_Investment  142.245
1552   1552 2025-05-01  Manufacturing_Investment  142.245
1553   1553 2025-06-01  Manufacturing_Investment  142.245
1554   1554 2025-07-01  Manufacturing_Investment  136.654
1555   1555 2025-08-01  Manufacturing_Investment  136.654
1556   1556 2025-09-01  Manufacturing_Investment  136.654
1557   1557 2025-10-01  Manufacturing_Investment  126.551
1558   1558 2025-11-01  Manufacturing_Investment  126.551
1559   1559 2025-12-01  Manufacturing_Investment  126.551
